### Step 1 - Import libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import warnings


warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

### Step - 2 Load data

In [5]:
rfm = pd.read_csv('../data/rfm_clv.csv')
df  = pd.read_csv('../data/clean_data.csv', parse_dates=['InvoiceDate'])


seg_colors = {'Champions': '#1D9E75', 'Loyal Customers': '#378ADD', 'Lost / Inactive': '#888780'}
print(f"Loaded: {len(rfm):,} customers across {rfm['Segment'].nunique()} segments")

Loaded: 5,878 customers across 3 segments


In [11]:
# RFM Data Preview
rfm.head(10)


,Customer ID,Recency,Frequency,Monetary,Cluster,Segment,predicted_purchases_12m,CLV_12m
0,12346,326,12,77556.46,1,Loyal Customers,2.471506,25962.373049
1,12347,2,8,5633.32,1,Loyal Customers,5.841563,3981.252162
2,12348,75,5,2019.40,1,Loyal Customers,3.244333,1413.954074
3,12349,19,4,4428.69,1,Loyal Customers,1.977177,2179.239819
4,12350,310,1,334.40,0,Lost / Inactive,0.000000,0.000000
5,12351,375,1,300.93,0,Lost / Inactive,0.000000,0.000000
6,12352,36,10,2849.84,1,Loyal Customers,6.742408,2161.109558
7,12353,204,2,406.76,0,Lost / Inactive,1.045127,108.757026
8,12354,232,1,1079.40,1,Loyal Customers,0.000000,0.000000
9,12355,214,2,947.61,1,Loyal Customers,0.815141,423.775904


In [12]:
# Clean Data Preview
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalRevenue,YearMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009-12
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,39.6,2009-12
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009-12
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,59.5,2009-12
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085,United Kingdom,30.6,2009-12
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085,United Kingdom,45.0,2009-12


### Insight 1 — Revenue concentration

In [13]:
# Group customers by segment and calculate key business metrics
seg_rev = rfm.groupby('Segment').agg(
    Customers=('Customer ID', 'count'),      # Number of customers in each segment
    Total_Revenue=('Monetary', 'sum'),       # Total historical revenue from each segment
    Avg_CLV=('CLV_12m', 'mean')              # Average predicted 12-month CLV per segment
)

# Round numeric values to 1 decimal place for cleaner output
seg_rev = seg_rev.round(1)

# Calculate each segment's share of total revenue
seg_rev['Revenue_%'] = (
    seg_rev['Total_Revenue'] / seg_rev['Total_Revenue'].sum() * 100
).round(1)

# Calculate each segment's share of total customers
seg_rev['Customer_%'] = (
    seg_rev['Customers'] / seg_rev['Customers'].sum() * 100
).round(1)

# Display the final segment revenue table
print("Revenue and customer breakdown by segment:")
print(seg_rev)

# Extract the Champions row for a focused business insight
champions = seg_rev.loc['Champions']

# Print a clear insight comparing customer share vs revenue share
print()
print(
    f"Champions: {champions['Customer_%']}% of customers "
    f"generate {champions['Revenue_%']}% of revenue"
)

# Business interpretation
print("This shows that a small high-value segment contributes a large share of revenue.")

Revenue and customer breakdown by segment:
                 Customers  Total_Revenue  Avg_CLV  Revenue_%  Customer_%
Segment                                                                  
Champions               24      3237553.9  55447.1       18.2         0.4
Lost / Inactive       2283      1240615.8     96.7        7.0        38.8
Loyal Customers       3571     13265259.5   2007.8       74.8        60.8

Champions: 0.4% of customers generate 18.2% of revenue
This shows that a small high-value segment contributes a large share of revenue.
